<a href="https://colab.research.google.com/github/dennistay1981/HKUST/blob/main/Lecture6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing libraries and data

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics

data=pd.read_csv('Lecture6.csv',index_col='Country')

data=pd.read_csv('https://raw.githubusercontent.com/dennistay1981/Resources/refs/heads/main/HG4054%20Language%20and%20Society%20Through%20Data%20Analytics/Lecture6.csv', index_col='Country')

Display all columns and rows

In [ ]:
pd.set_option('display.max_rows',None)
pd.set_option('display.max_columns',None)
pd.set_option('display.width', 1000)


Visualizing the data

In [ ]:
sns.scatterplot(data,y="Happiness",x="GDP_pc",hue='Regime')

#log transform GDP_pc and plot again. Note that np.log takes the natural logarithm; i.e. base=Euler's number
data['GDP_log']=np.log(data.GDP_pc)
sns.scatterplot(data,y="Happiness",x="GDP_log",hue='Regime')


Algorithm 1: K-Nearest Neighbors

In [40]:
# Define outcome and predictors
y = data['Regime']
x = data[['Happiness','GDP_log']]


# Grid search
# Setup arrays to store accuracy values.
neighbors = np.arange(1, 16)
accuracy = np.empty(len(neighbors))

# Loop over different values of k, fit model, and compute accuracy
for i, k in enumerate(neighbors):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(x,y)
    accuracy[i] = knn.score(x, y)

# Generate plot
plt.title('k-NN: Varying Number of Neighbors')
plt.plot(neighbors, accuracy)
plt.xticks(neighbors)
plt.xlabel('Number of Neighbors')
plt.ylabel('Accuracy')
plt.show()



# Create a kNN classifier and fit it to data. If n_neigbors is not specified, the default value=5
knn = KNeighborsClassifier(n_neighbors = 3)
knn.fit(x,y)


# Predict the y labels and add to dataset
knn.predict(x)
data['predicted']=knn.predict(x)
data[['Regime','predicted']]

# Evaluate accuracy
knn.score(x, y)


0.7039473684210527

Visualizing outcomes: confusion matrix

In [ ]:
#Confusion matrix with raw frequencies and percentages
cnf_matrix = metrics.confusion_matrix(data['Regime'], data['predicted'])  #real, then predicted
cnf_matrix


Visualizing outcomes: heatmap

In [ ]:
#Heatmap with actual numbers (rows=actual labels, columns=predicted labels)
labels = data['Regime'].unique()   #obtain labels in correct order

sns.heatmap(cnf_matrix, annot=True, cmap="Blues", yticklabels=labels, xticklabels=labels, annot_kws={"size": 15})
plt.ylabel('Actual label')
plt.xlabel('Predicted label')

#Heatmap with percentages (rows=actual labels, columns=predicted labels)
sns.heatmap(cnf_matrix/np.sum(cnf_matrix), annot=True, cmap="Blues",fmt='.2%', yticklabels=labels, xticklabels=labels, annot_kws={"size": 15})
plt.ylabel('Actual label')
plt.xlabel('Predicted label')

Visualizing outcomes: classification report

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(data['Regime'],data['predicted']))


Visualizing outcomes: scatterplots

In [ ]:
#Scatterplots of actual vs. predicted labels
sns.scatterplot(data, y='Happiness', x='GDP_log', hue='Regime', hue_order=['Authoritarian', 'Hybrid', 'Flawed', 'Full'], s=80)
plt.title("Actual labels")

sns.scatterplot(data, y='Happiness', x='GDP_log', hue='predicted', hue_order=['Authoritarian', 'Hybrid', 'Flawed', 'Full'], s=80)
plt.title("Predicted labels")


Algorithm 2: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Apply base-10 logarithmic transformation to GDP per capita
data['GDP_log'] = np.log10(data['GDP_pc'])

# Remove missing values for the relevant columns
data_clean = data.dropna(subset=['Happiness', 'GDP_log', 'Regime'])

# Define predictors and target
X = data_clean[['Happiness', 'GDP_log']]
y = data_clean['Regime']

# Initialize and fit the Logistic Regression model
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X, y)

# Generate predictions on the same dataset to evaluate in-sample fit
y_pred = logreg.predict(X)


# Show predicted probabilties of each regime type for each country
probabilities = logreg.predict_proba(X)

results = data_clean[['Regime']].copy()
results = results.reset_index()

if 'index' in results.columns:
    results = results.rename(columns={'index': 'Country'})

results['Predicted_Regime'] = y_pred

# Loop through the model classes to label the probability columns
for index, category_name in enumerate(logreg.classes_):
    column_label = 'Prob_' + str(category_name)
    results[column_label] = probabilities[:, index]
results



# Create the confusion matrix
cnf_matrix = metrics.confusion_matrix(y, y_pred, labels=logreg.classes_)  #real, then predicted
cnf_matrix

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap='Blues', ax=ax, xticks_rotation=45)
ax.set_title('Confusion Matrix Logistic Regression')
plt.tight_layout()

# Print accuracy
accuracy = logreg.score(X, y)
print(f"In-sample Accuracy {accuracy:.4f}")
print("Classes", logreg.classes_)

# Print classification report
print(classification_report(data['Regime'],y_pred))


Extracting 'boundary' countries

In [ ]:
data.loc[(data['Regime'] == 'Flawed') & (data['predicted'] =='Hybrid')]
data.loc[(data['Regime'] == 'Hybrid') & (data['predicted'] =='Flawed')]